# **Lab 02 - Introduction to Q-learning**

##### Copyright by UIT-NC@NT549

## **Some instructions before getting started**:
<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">

Start the Kernel: At the top right, choose <strong>Select Kernel ➞ Python Environments...</strong>

You can run all code blocks to check: From the menu bar, choose <strong>Run All</strong>.

Complete all code blocks marked with the comment <span style="font-family: monospace; font-weight: bold; color:white; background-color: green;"> ### YOU NEED TO WRITE YOUR CODE BELOW ### </span>
</div>

## Part 1: Implementing Q-learning with the FrozenLake-v1 environment

In [ ]:
# import Gymnasium library and alias as gym
import gymnasium as gym
import random

In [ ]:
# ============================================================
# Q-learning agent for Gymnasium FrozenLake-v1
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

class QLearningAgent:
    def __init__(self, env, alpha=0.1, gamma=0.9):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.q_table = {}

    def get_q_value(self, state, action):
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        return self.q_table.get((state, action), 0.0)

    def update_q_value(self, state, action, reward, next_state):
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        best_next_q = max(
            self.get_q_value(next_state, a) for a in range(self.env.action_space.n)
        )
        current_q = self.get_q_value(state, action)
        target_q = reward + self.gamma * best_next_q
        new_q = (1 - self.alpha) * current_q + self.alpha * target_q
        self.q_table[(state, action)] = new_q

    def choose_action(self, state, epsilon=0.1, policy="q_learning"):
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        # Random policy option
        if policy == "random":
            return self.env.action_space.sample()

        # Exploration
        if random.random() < epsilon:
            return self.env.action_space.sample()

        # Exploitation
        q_values = [self.get_q_value(state, a) for a in range(self.env.action_space.n)]
        max_q = max(q_values)
        best_actions = [a for a, q in enumerate(q_values) if q == max_q]
        return random.choice(best_actions)

    def choose_greedy_action(self, state):
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        q_values = [self.get_q_value(state, a) for a in range(self.env.action_space.n)]
        max_q = max(q_values)
        best_actions = [a for a, q in enumerate(q_values) if q == max_q]
        return random.choice(best_actions)


def export_q_table_to_txt(agent, file_path="q_table_final.txt"):
    """Export all learned Q-values to a CSV-like text file."""
    with open(file_path, "w", encoding="utf-8") as f:
        f.write("State,Action,Q-value\n")
        for (state, action) in sorted(agent.q_table.keys()):
            q_val = agent.q_table[(state, action)]
            f.write(f"{state},{action},{q_val:.6f}\n")
    print(f"Q-table exported to: {file_path}")


def show_final_result_text(agent):
    """Run one full evaluation episode in ANSI text mode."""
    demo_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")
    state, _ = demo_env.reset()

    print("=== FINAL RUN WITH GREEDY POLICY (TEXT) ===")
    print(demo_env.render())

    done = False
    step = 0
    reward = 0

    while not done:
        action = agent.choose_greedy_action(state)
        next_state, reward, terminated, truncated, _ = demo_env.step(action)
        done = terminated or truncated
        step += 1
        print(f"Step {step} - Action: {action}, Reward: {reward}")
        print(demo_env.render())
        state = next_state

    print("Agent reached the goal!" if reward > 0 else "Agent failed to reach the goal.")
    demo_env.close()


def show_final_result_pygame(agent, delay_seconds=0.5):
    """
    Run one full evaluation episode with GUI rendering ("human" mode).
    Note: Requires a local environment with display support (pygame).
    """
    import time

    demo_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human")
    state, _ = demo_env.reset()

    print("=== FINAL RUN WITH GREEDY POLICY (PYGAME/HUMAN) ===")
    done = False
    step = 0
    reward = 0

    while not done:
        action = agent.choose_greedy_action(state)
        next_state, reward, terminated, truncated, _ = demo_env.step(action)
        done = terminated or truncated
        step += 1

        print(f"Step {step} - Action: {action}, Reward: {reward}")
        state = next_state
        time.sleep(delay_seconds)

    print("Agent reached the goal!" if reward > 0 else "Agent failed to reach the goal.")
    time.sleep(1.0)
    demo_env.close()


# ============================================================
# Training routine
# ============================================================

if __name__ == "__main__":
    # Create deterministic FrozenLake environment for training
    ### YOU NEED TO WRITE YOUR CODE BELOW ###
    env = gym.make("FrozenLake-v1", is_slippery=False)

    # Initialize Q-learning agent with default hyperparameters
    ### YOU NEED TO WRITE YOUR CODE BELOW ###
    agent = QLearningAgent(env, alpha=0.1, gamma=0.9)

    num_episodes = 1000

    # Main training loop
    for _ in range(num_episodes):
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        state, _ = env.reset()
        done = False

        while not done:
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            action = agent.choose_action(state, epsilon=0.1)

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            agent.update_q_value(state, action, reward, next_state)

            state = next_state

    # Print learned Q-table entries
    print("Training completed. Q-table:")
    for (state, action) in sorted(agent.q_table.keys()):
        q_val = agent.q_table[(state, action)]
        print(f"State: {state}, Action: {action}, Q-value: {q_val:.2f}")

    # Save Q-table to file
    export_q_table_to_txt(agent, "q_table_final.txt")

    # Show one text-based demonstration run
    show_final_result_text(agent)

    # Show one GUI-based demonstration run (cần có display/pygame)
    show_final_result_pygame(agent, delay_seconds=0.4)

    # ============================================================
    # YEU CAU 1: So sanh Q-learning vs Random Policy
    # ============================================================
    eval_episodes = 100
    q_eval_rewards = []
    random_eval_rewards = []

    for _ in range(eval_episodes):
        # Q-learning greedy evaluation
        state, _ = env.reset()
        done = False
        total_r = 0
        while not done:
            act = agent.choose_greedy_action(state)
            next_state, reward, terminated, truncated, _ = env.step(act)
            done = terminated or truncated
            total_r += reward
            state = next_state
        q_eval_rewards.append(total_r)

        # Random evaluation
        state, _ = env.reset()
        done = False
        total_r = 0
        while not done:
            act = agent.choose_action(state, policy="random")
            next_state, reward, terminated, truncated, _ = env.step(act)
            done = terminated or truncated
            total_r += reward
            state = next_state
        random_eval_rewards.append(total_r)

    plt.figure(figsize=(10, 5))
    plt.plot(q_eval_rewards, label="Q-learning Policy", color="blue", alpha=0.7)
    plt.plot(random_eval_rewards, label="Random Policy", color="red", alpha=0.7)
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.title("FrozenLake: Q-learning vs Random Policy")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("part1_qlearning_vs_random.png", dpi=150)
    plt.show()

    print(f"\nQ-learning avg reward: {np.mean(q_eval_rewards):.2f}")
    print(f"Random avg reward:     {np.mean(random_eval_rewards):.2f}")

    # ============================================================
    # YEU CAU 4: So sanh 3 bo sieu tham so
    # ============================================================
    cases = [
        (0.5, 0.2, 0.1, "Case1 (α=0.5, γ=0.2, ε=0.1)"),
        (0.1, 0.95, 0.2, "Case2 (α=0.1, γ=0.95, ε=0.2)"),
        (0.1, 0.8, 0.7, "Case3 (α=0.1, γ=0.8, ε=0.7)"),
    ]

    plt.figure(figsize=(10, 5))
    for alpha_val, gamma_val, eps_val, label in cases:
        case_env = gym.make("FrozenLake-v1", is_slippery=False)
        case_agent = QLearningAgent(case_env, alpha=alpha_val, gamma=gamma_val)
        case_rewards = []
        for ep in range(1000):
            s, _ = case_env.reset()
            d = False
            tr = 0
            while not d:
                a = case_agent.choose_action(s, epsilon=eps_val)
                ns, r, term, trunc, _ = case_env.step(a)
                d = term or trunc
                case_agent.update_q_value(s, a, r, ns)
                s = ns
                tr += r
            case_rewards.append(tr)
        case_env.close()
        smoothed = pd.Series(case_rewards).rolling(50, min_periods=1).mean()
        plt.plot(smoothed, label=label, alpha=0.8)

    plt.xlabel("Episode")
    plt.ylabel("Reward (smoothed)")
    plt.title("FrozenLake: Convergence with Different Hyperparameters")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("part1_hyperparams.png", dpi=150)
    plt.show()

    env.close()

## Part 2: Q-learning with Custom Environment "VacuumCleaner"

In [ ]:
# Build a simple custom Gymnasium environment named "VacuumCleaner-v0".
import numpy as np
import os
import time
from IPython.display import clear_output

class VacuumCleanerEnv(gym.Env):
    def __init__(self, m=5, n=5, obstacle=(2, 2)):
        super(VacuumCleanerEnv, self).__init__()
        self.m = m
        self.n = n
        self.obstacle = tuple(obstacle)

        # Action space: 0=up, 1=down, 2=left, 3=right
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.action_space = gym.spaces.Discrete(4)

        # Observation space: position and dust grid
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.observation_space = gym.spaces.Dict({
            'position': gym.spaces.MultiDiscrete([m, n]),
            'dust': gym.spaces.MultiBinary([m, n])
        })

        self.reset()

    def _encode_state(self):
        """Encode state as hashable tuple for Q-table."""
        pos = (int(self.position[0]), int(self.position[1]))
        dust = tuple(self.dust_grid.flatten().astype(int))
        return (pos, dust)

    def reset(self, *, seed=None, options=None):
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.position = np.array([0, 0])
        self.dust_grid = np.ones((self.m, self.n), dtype=np.int8)
        self.dust_grid[self.obstacle] = 0
        self.dust_grid[0, 0] = 0  # Starting position is cleaned
        self.total_reward = 0.0
        self.truncated = False
        self.terminated = False
        obs = {'position': self.position.copy(), 'dust': self.dust_grid.copy()}
        return obs, {}

    def step(self, action):
        # compute candidate new position
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        if action == 0:   # Up
            candidate = self.position + np.array([-1, 0])
        elif action == 1: # Down
            candidate = self.position + np.array([1, 0])
        elif action == 2: # Left
            candidate = self.position + np.array([0, -1])
        elif action == 3: # Right
            candidate = self.position + np.array([0, 1])
        else:
            candidate = self.position.copy()

        # boundary check
        if (0 <= candidate[0] < self.m) and (0 <= candidate[1] < self.n):
            if tuple(candidate) == self.obstacle:
                self.position = candidate.copy()
                reward = -10.0
                self.terminated = True
                obs = {'position': self.position.copy(), 'dust': self.dust_grid.copy()}
                self.total_reward += reward
                return obs, reward, True, False, {}
            else:
                self.position = candidate.copy()

        # Clean dust logic
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        x, y = int(self.position[0]), int(self.position[1])
        if self.dust_grid[x, y] == 1:
            reward = 1.0
            self.dust_grid[x, y] = 0
        else:
            reward = -0.5

        self.total_reward += reward

        # check if all cleaned
        if np.sum(self.dust_grid) == 0:
            reward += 10.0
            self.total_reward += 10.0
            self.terminated = True

        obs = {'position': self.position.copy(), 'dust': self.dust_grid.copy()}
        return obs, reward, bool(self.terminated), bool(self.truncated), {}

    def render(self, mode='human'):
        try:
            clear_output(wait=True)
        except Exception:
            os.system('cls' if os.name == 'nt' else 'clear')
        display = np.full((self.m, self.n), ' ', dtype='<U1')
        for i in range(self.m):
            for j in range(self.n):
                if (i, j) == self.obstacle:
                    display[i, j] = '#'
                elif self.dust_grid[i, j] == 1:
                    display[i, j] = '.'
                else:
                    display[i, j] = ' '
        x, y = int(self.position[0]), int(self.position[1])
        if (x, y) == self.obstacle:
            display[x, y] = 'X'
        else:
            display[x, y] = 'R'
        for row in display:
            print(''.join(row))
        print(f"Total reward: {self.total_reward}")
        time.sleep(0.15)

In [ ]:
def robot_policy(option="random", env=None):
     """
     A simple policy function that selects an action based on the specified option.
     Currently supports only a random policy.
     """
     if option == "random":
          return env.action_space.sample()  # Randomly select an action from the action space
     

In [ ]:
if __name__ == "__main__":
    env = VacuumCleanerEnv(m=5, n=5, obstacle=(2, 2))
    obs, _ = env.reset()
    env.render()
    terminated = False
    truncated = False

    while not terminated and not truncated:
        action = robot_policy(option="random", env=env)
        obs, reward, terminated, truncated, info = env.step(action)
        env.render()
        print(f"Action: {action}, Reward: {reward}, Terminated: {terminated}")
        if terminated or truncated:
            print("Episode finished with total reward:", env.total_reward)
            break

In [ ]:
def save(...):
    pass

Evaluation and Analysis

## Part 3: Q-learning with Load Balancing Problem

In [ ]:
"""
Load Balancing Environment Simulation
"""
import random as rand_module
from collections import deque

class Task:
    def __init__(self, task_id: int, processing_time: int):
        self.task_id = task_id
        self.processing_time = processing_time


class Server:
    def __init__(self, queue_capacity: int):
        self.queue_capacity = queue_capacity
        self.queue = deque()
        self.current_task = None
        self.remaining_time = 0

    def run_one_step(self):
        completed_task = None
        # Process one time unit for currently running task
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        if self.current_task is not None:
            self.remaining_time -= 1
            if self.remaining_time <= 0:
                completed_task = self.current_task
                self.current_task = None
                self.remaining_time = 0

        # If server becomes idle, immediately pull next task from queue
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        if self.current_task is None and len(self.queue) > 0:
            next_task = self.queue.popleft()
            self.current_task = next_task
            self.remaining_time = next_task.processing_time

        return completed_task

    def add_task(self, task: Task) -> bool:
        # Start immediately if server is free
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        if self.current_task is None:
            self.current_task = task
            self.remaining_time = task.processing_time
            return True

        # Otherwise enqueue if capacity allows
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        if len(self.queue) < self.queue_capacity:
            self.queue.append(task)
            return True

        # Queue full -> task dropped
        return False

    def queue_length(self) -> int:
        return len(self.queue)

In [ ]:
class LoadBalancingEnv(gym.Env):
    metadata = {"render.modes": ["human"]}

    def __init__(self, n_servers: int = 3, queue_capacity: int = 2, seed: int = None):
        super().__init__()
        self.n_servers = n_servers
        self.queue_capacity = queue_capacity
        self.rng = rand_module.Random(seed)
        self.servers = [Server(queue_capacity) for _ in range(n_servers)]
        self.time = 0
        self.total_reward = 0.0
        self.next_task_id = 0
        self.tasks_created = {}
        self.tasks_completed = set()
        self.tasks_dropped = set()
        self.tasks_accepted = set()

        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.action_space = gym.spaces.Discrete(n_servers)
        self.observation_space = gym.spaces.Dict({
            "servers": gym.spaces.Tuple(tuple(
                gym.spaces.Dict({
                    "remaining_time": gym.spaces.Box(0, 100, shape=()),
                    "queue_length": gym.spaces.Discrete(queue_capacity + 1)
                }) for _ in range(n_servers)
            )),
            "time": gym.spaces.Box(0, float('inf'), shape=())
        })

    def _encode_state(self):
        """Encode state as hashable tuple for Q-table."""
        server_states = tuple(
            (min(int(s.remaining_time), 5), s.queue_length())
            for s in self.servers
        )
        return server_states

    def _new_task(self) -> Task:
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        t = Task(self.next_task_id, self.rng.randint(1, 5))
        self.tasks_created[t.task_id] = t
        self.next_task_id += 1
        return t

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self.rng.seed(seed)
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.servers = [Server(self.queue_capacity) for _ in range(self.n_servers)]
        self.time = 0
        self.total_reward = 0.0
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.next_task_id = 0
        self.tasks_created = {}
        self.tasks_completed = set()
        self.tasks_dropped = set()
        self.tasks_accepted = set()
        return self._get_observation(), {}

    def step(self, action: int):
        reward = 0.0

        # 1) Process running tasks on each server
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        for server in self.servers:
            completed = server.run_one_step()
            if completed is not None:
                self.tasks_completed.add(completed.task_id)
                reward += 2.0

        # 2) Generate one new incoming task
        new_task = self._new_task()

        # 3) Route task based on action
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        accepted = self.servers[action].add_task(new_task)
        if accepted:
            self.tasks_accepted.add(new_task.task_id)
            reward += 0.5
        else:
            self.tasks_dropped.add(new_task.task_id)
            reward -= 5.0

        # 4) Add congestion penalty
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        for server in self.servers:
            reward -= 0.5 * server.queue_length()

        self.total_reward += reward
        self.time += 1

        terminated = False
        truncated = False
        info = self._get_info()
        return self._get_observation(), reward, terminated, truncated, info

    def _get_observation(self):
        return {
            "servers": tuple(
                {
                    "remaining_time": float(server.remaining_time),
                    "queue_length": server.queue_length(),
                }
                for server in self.servers
            ),
            "time": float(self.time),
        }

    def _get_info(self):
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        total_created = len(self.tasks_created)
        dropped = len(self.tasks_dropped)
        completed = len(self.tasks_completed)
        accepted = len(self.tasks_accepted)

        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        drop_rate = dropped / total_created if total_created > 0 else 0.0
        completion_rate = completed / total_created if total_created > 0 else 0.0

        return {
            "time": self.time,
            "total_created": total_created,
            "accepted": accepted,
            "dropped": dropped,
            "completed": completed,
            "drop_rate": drop_rate,
            "completion_rate": completion_rate,
            "total_reward": self.total_reward,
        }

In [ ]:
def load_balancing_policy(option="random", env=None, step=0):
    """Baseline policies for Load Balancing."""
    if option == "random":
        return env.action_space.sample()
    ### YOU NEED TO WRITE YOUR CODE BELOW ###
    if option == "round_robin":
        return step % env.n_servers
    raise ValueError(f"Unsupported policy option: {option}")

In [ ]:
if __name__ == "__main__":
    import pandas as pd
    env = LoadBalancingEnv(n_servers=3, queue_capacity=2, seed=42)
    obs, info = env.reset()

    metrics_list = []
    for step in range(20):
        action = load_balancing_policy(option="random", env=env)
        obs, reward, terminated, truncated, info = env.step(action)
        
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        print(f"Step {step}: action={action}, reward={reward:.2f}, info={info}")
        metrics_list.append(info)

    ### YOU NEED TO WRITE YOUR CODE BELOW ###
    df = pd.DataFrame(metrics_list)
    df.to_csv("lb_metrics_random.csv", index=False)
    print("\nSummary:")
    print(f"Total created: {info['total_created']}")
    print(f"Dropped: {info['dropped']}")
    print(f"Completed: {info['completed']}")
    print(f"Drop rate: {info['drop_rate']:.4f}")
    print(f"Total reward: {info['total_reward']:.2f}")

In [ ]:
def save_metrics(...):
     # Placeholder for a function to save metrics
     pass

Evaluation and Analysis

## CONGRATULATIONS TEAM!

Congratulations to the team for completing Lab02 - Introduction to Q-learning.
Keep up the effort in the next sections.

References: https://gymnasium.farama.org/ 

## ADDITIONAL INFORMATION

**Author**: M.Sc. Phan Trung Phat - Department of Computer Networks and Communications, UIT

**Contact**: phatpt@uit.edu.vn
